# TIM Example 2 — Output compare and input capture

**Target:** STM32F103C8T6 · TIM2_CH1 / PA0  
**Focus:** Hardware timestamping at the output or input pin.

## Feature classification

| Timer function | Channel direction | Modes / variations | Result |
|---|---|---|---|
| Output compare | Timer → pin/event | Frozen, active, inactive, toggle, forced; PWM is a specialized compare mode | Compare event when CNT matches CCRx |
| Input capture | Pin → timer | Rising, falling, both; input filter and prescaler | Capture CNT into CCRx on edge |
| Timing pair | Two captures / one channel | Period or pulse-width measurement | Subtract timestamps, handle wrap |

Default TIM2 routes CH1–CH4 to PA0–PA3. PA0 is also ADC1_IN0 / WKUP; only one function can own it at a time. Configure the channel pin and timer using CubeMX.

## Repository bridge — update interrupt vs. channel capture

The project `STM32F103C8T6_Interrupt_push_buuton` starts TIM4 update interrupts immediately. PSC=7200 and ARR=999 produce about 10 Hz from a 72 MHz TIM4 clock (the exact divisor is 7201 × 1000). It is an update-interrupt example, not an input-capture example. Its timer callback toggles LEDs directly, and its EXTI callback does the same; refactor the callbacks to set event flags before extending this lesson to capture callbacks.

## Output compare toggle example

Set TIM2_CH1 to output compare toggle and enable the channel output on PA0. The output changes state at each compare match. With one compare event per counter cycle, toggle output frequency is approximately `f_TIM / (2 × (PSC+1) × (ARR+1))`. PWM mode is more appropriate for a continuous duty-cycle waveform.

```c
HAL_TIM_OC_Start(&htim2, TIM_CHANNEL_1);
// Set CCR1 to the desired compare count using __HAL_TIM_SET_COMPARE().
```


## Input capture example: measure period

Configure PA0 / TIM2_CH1 as direct input capture on rising edges, with a suitable digital filter for the signal. Start capture with interrupt or DMA. On successive captures, subtract the previous count; if the 16-bit counter wrapped, unsigned 16-bit subtraction produces the modulo-65536 interval. The timer clock and prescaler convert counts to seconds.

At 72 MHz timer clock with prescaler 71, the counter ticks at 1 MHz (1 µs/count). The first capture only seeds the timestamp; subsequent captures produce intervals. Modulo subtraction handles at most one counter wrap between edges; use overflow tracking if a longer interval is possible.

```c
static volatile uint16_t previous_capture;
static volatile uint16_t period_ticks;
static volatile uint8_t capture_ready;
static uint8_t have_previous_capture;

void HAL_TIM_IC_CaptureCallback(TIM_HandleTypeDef *htim)
{
    if (htim->Instance == TIM2 && htim->Channel == HAL_TIM_ACTIVE_CHANNEL_1) {
        uint16_t current = (uint16_t)HAL_TIM_ReadCapturedValue(htim, TIM_CHANNEL_1);
        if (have_previous_capture != 0U) {
            period_ticks = (uint16_t)(current - previous_capture);
            capture_ready = 1U;
        } else {
            have_previous_capture = 1U;
        }
        previous_capture = current;
    }
}

// Start with HAL_TIM_IC_Start_IT(&htim2, TIM_CHANNEL_1).
```


## Pulse-width and input-mode variations

- **Period:** Two rising-edge captures; frequency is `f_counter / delta_ticks`.
- **High pulse width:** Rising then falling capture (or F1 timer PWM-input/reset-slave configuration) and subtract timestamps.
- **Noise rejection:** Input filter requires a stable digital level for multiple timer samples; it is not a substitute for electrical conditioning.
- **DMA:** Capture DMA reduces per-edge CPU work for high-rate streams, but buffer overflow and ownership must be handled.

For both-edge timing, track edge polarity/state explicitly or use the timer's PWM-input mode; account for wrap and ensure the signal frequency does not exceed the capture/processing rate.